<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-16b-reporting-results-seeds-ablations-and-a-datasheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Discussion 16 (continued) - From a result to something others can trust: seeds, ablations, honest statistics and a datasheet

A capstone ends with a number. A research result, or a benchmark you release for others to use, has to survive a sceptical reader: *would a different random seed give the same answer, which part of the system actually produced the gain, how sure are you, and what would someone need to repeat it?* This lab runs one small, realistic study end to end and writes it up the way a paper's results section and a dataset's datasheet would.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded; not needed for the certificate. Runs on the free Colab CPU in about five minutes.

*Original code written for this course. Data: the Banking77 customer-intent dataset (PolyAI, CC BY 4.0; Casanueva et al., 2020), loaded from a public copy, with sentence embeddings from all-MiniLM-L6-v2. The study is deliberately small so that it runs fast; the discipline is what transfers.*

In [ ]:
%pip install -q sentence-transformers scikit-learn pandas pyarrow scipy numpy

In [ ]:
import time, json, hashlib, platform, itertools
import numpy as np, pandas as pd, scipy, sklearn
from scipy import stats
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)      # the MLPs hit the iteration cap; reported as a limitation below
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer
BASE = "https://huggingface.co/datasets/mteb/banking77/resolve/main/data/"
train = pd.read_parquet(BASE + "train-00000-of-00001.parquet"); test = pd.read_parquet(BASE + "test-00000-of-00001.parquet")
rng0 = np.random.default_rng(0)
INTENTS = sorted(rng0.choice(train.label_text.unique(), 20, replace=False))
tr = train[train.label_text.isin(INTENTS)].groupby("label_text").sample(n=20, random_state=0).reset_index(drop=True)      # 20 labelled messages per intent
te = test[test.label_text.isin(INTENTS)].reset_index(drop=True)
enc = SentenceTransformer("all-MiniLM-L6-v2")
Xtr = enc.encode(list(tr.text), normalize_embeddings=True); Xte = enc.encode(list(te.text), normalize_embeddings=True)
print(len(tr), "training and", len(te), "test messages;", len(INTENTS), "intents")

## Part 1 - The study

A question a team might ask: *does a small neural classifier on top of the embeddings beat plain logistic regression, and does a regulariser (weight decay) help it?* Three systems:

- **A**: logistic regression (the baseline);
- **B**: a one-hidden-layer network (256 units);
- **C**: the same network with weight decay.

The network's result depends on its random initialisation and on the order of the training data, so we must not judge it from one run.

In [ ]:
def run(system, seed):
    if system == "A": clf = LogisticRegression(max_iter=2000, C=10, random_state=seed)
    elif system == "B": clf = MLPClassifier((256,), max_iter=300, random_state=seed, alpha=1e-6)
    else: clf = MLPClassifier((256,), max_iter=300, random_state=seed, alpha=1e-1)
    clf.fit(Xtr, tr.label_text); return float((clf.predict(Xte) == te.label_text.to_numpy()).mean())

# the tempting version: one run each
single = {s: run(s, 0) for s in "ABC"}
print("single seed (seed 0):", {k: round(v, 4) for k, v in single.items()})

One run says "C is best, B is second, A is last" (or some other order). The next cell repeats every system over 20 seeds.

In [ ]:
SEEDS = list(range(20))
acc = {s: np.array([run(s, sd) for sd in SEEDS]) for s in "ABC"}
print(f"{'system':8}{'mean':>9}{'std':>9}{'min':>9}{'max':>9}")
for s in "ABC": print(f"{s:8}{acc[s].mean():>9.4f}{acc[s].std(ddof=1):>9.4f}{acc[s].min():>9.4f}{acc[s].max():>9.4f}")
print("\nhow often the single-seed ranking would have matched the 20-seed mean ranking, over all 20 possible 'single seeds':")
mean_rank = sorted("ABC", key=lambda s: -acc[s].mean())
match = np.mean([sorted("ABC", key=lambda s: -acc[s][i]) == mean_rank for i in range(len(SEEDS))])
print(f"   {match:.0%} of the time (mean ranking: {' > '.join(mean_rank)})")

**Read it.** System A has a spread of exactly zero because logistic regression with this solver does not depend on the seed, so its "20 runs" are one run repeated; only the networks vary. The spread between seeds (the *std* column) tells you how large a difference you may believe. Whenever the gap between two systems is comparable to the seed-to-seed spread, a single run can rank them either way, and the last line shows how often a one-seed ranking would have agreed with the many-seed ranking.

## Part 2 - Report a difference with its uncertainty

For two systems run on the same seeds and the same test messages, the standard report is the mean difference with a confidence interval and an effect size, not a bare "B beat A". Because the runs are paired by seed, we use a paired t-interval; we also give a bootstrap interval as a check that does not assume normality. Three pairwise comparisons means three tests, so we apply the **Holm correction** to the p-values (without it, running enough comparisons will eventually produce a "significant" one).

In [ ]:
pairs = [("B", "A"), ("C", "A"), ("C", "B")]
rows = []
rng = np.random.default_rng(1)
for x, y in pairs:
    d = acc[x] - acc[y]; n = len(d)
    se = d.std(ddof=1) / np.sqrt(n); tcrit = stats.t.ppf(0.975, n - 1)
    boot = [d[rng.integers(0, n, n)].mean() for _ in range(5000)]
    t, p = stats.ttest_1samp(d, 0.0)
    rows.append(dict(comparison=f"{x} - {y}", mean_diff=d.mean(), ci_t=(d.mean() - tcrit * se, d.mean() + tcrit * se),
                     ci_boot=(np.percentile(boot, 2.5), np.percentile(boot, 97.5)), cohens_d=d.mean() / d.std(ddof=1), p_raw=p))
order = np.argsort([r["p_raw"] for r in rows]); m = len(rows); running = 0.0
for rank, i in enumerate(order):                                      # Holm step-down: (m - rank) * p, made monotone
    running = max(running, min(1.0, (m - rank) * rows[i]["p_raw"])); rows[i]["p_holm"] = running
print(f"{'comparison':12}{'mean diff':>11}{'95% CI (paired t)':>24}{'95% CI (bootstrap)':>24}{'Cohen d':>9}{'p raw':>10}{'p Holm':>10}")
for r in rows:
    print(f"{r['comparison']:12}{r['mean_diff']:>+11.4f}   [{r['ci_t'][0]:+.4f}, {r['ci_t'][1]:+.4f}]   [{r['ci_boot'][0]:+.4f}, {r['ci_boot'][1]:+.4f}]{r['cohens_d']:>9.2f}{r['p_raw']:>10.4f}{r['p_holm']:>10.4f}")

**Read it.** State a finding only with its interval. An interval that includes zero means the study did not show a difference; it does *not* show that there is none (the sample may simply be too small to see it). The effect size (Cohen's d, the mean difference in units of its own spread) separates "statistically detectable" from "large enough to matter": with many seeds a tiny difference can be significant. And note what the 20 seeds do *not* cover: the test set is one fixed sample of messages, so the results also carry test-sampling uncertainty (Chapter 8 of the agents course and Chapter 9 here show how to interval that).

## Part 3 - An ablation: which part of the system produced the gain?

If C beats A, is it the hidden layer or the regulariser? An **ablation** removes one component at a time. The three systems above already form one (A has neither, B has the layer, C has both); we line them up so the contribution of each step is explicit, with intervals.

In [ ]:
steps = [("baseline A (logistic regression)", acc["A"], None), ("+ hidden layer (B)", acc["B"], acc["A"]), ("+ weight decay (C)", acc["C"], acc["B"])]
print(f"{'configuration':36}{'mean accuracy':>15}{'change from the step above':>34}")
for name, a, prev in steps:
    if prev is None: print(f"{name:36}{a.mean():>15.4f}{'-':>34}")
    else:
        d = a - prev; se = d.std(ddof=1) / np.sqrt(len(d)); h = stats.t.ppf(0.975, len(d) - 1) * se
        print(f"{name:36}{a.mean():>15.4f}{d.mean():>+17.4f} [{d.mean() - h:+.4f}, {d.mean() + h:+.4f}]")

## Part 3b - The other source of uncertainty: the test set

Twenty seeds measure how much the *training* randomness moves the result. They say nothing about how much the result would move if the test messages had been a different sample of the same kind. Resampling the test messages (a bootstrap over messages) measures that, and for a *paired* difference it uses the same messages for both systems.

In [ ]:
def correct_vec(system, seed):
    if system == "A": clf = LogisticRegression(max_iter=2000, C=10, random_state=seed)
    elif system == "B": clf = MLPClassifier((256,), max_iter=300, random_state=seed, alpha=1e-6)
    else: clf = MLPClassifier((256,), max_iter=300, random_state=seed, alpha=1e-1)
    clf.fit(Xtr, tr.label_text); return (clf.predict(Xte) == te.label_text.to_numpy()).astype(float)
V = {s: np.mean([correct_vec(s, sd) for sd in range(5)], axis=0) for s in "ABC"}        # per-message correctness, averaged over 5 seeds
rb = np.random.default_rng(3); n = len(te); idx = [rb.integers(0, n, n) for _ in range(4000)]
print(f"test set: {n} messages, so one message is worth {100 / n:.3f} percentage points")
for s_ in "ABC":
    b = [V[s_][i].mean() for i in idx]; print(f"  system {s_}: accuracy {V[s_].mean():.4f}, 95% interval over test messages [{np.percentile(b, 2.5):.4f}, {np.percentile(b, 97.5):.4f}]")
for x, y in pairs:
    d = V[x] - V[y]; b = [d[i].mean() for i in idx]
    print(f"  {x} - {y}: {d.mean():+.4f}, paired interval over test messages [{np.percentile(b, 2.5):+.4f}, {np.percentile(b, 97.5):+.4f}]")

**Read it, and compare with Part 2.** In the run shown, the seed analysis called two of the three differences significant (B below A, and C above B, both with Holm-corrected p below 0.002) and reported them with narrow intervals and large effect sizes. The absolute differences are 0.3 to 0.5 percentage points: two to four test messages out of 799. Resampling the *messages* tells a different story: the paired intervals for B minus A and C minus A include zero, and the interval for C minus B only just excludes it (about +0.1 to +0.8 points). Both views are true. The seed analysis says the ranking of the networks is stable *given this test set*, because training randomness here is tiny; the test-set analysis says you cannot be confident the same ranking would hold on new messages. A large effect size on a tiny absolute difference is statistically detectable and practically negligible. Report both sources of uncertainty, and say which one your interval covers.

**Read it.** Each row tells you what that single step bought, on this data. An ablation also protects against the common error of crediting an improvement to the new idea when a neighbouring change (more training, a better learning rate) produced it: the rule is that **everything except the studied component is held fixed**, including the budget for tuning each variant. Here the hyper-parameters were not tuned per variant; say so, because tuning one variant harder than another is the most common way a result is inflated.

## Part 4 - The record: manifest, results table and a datasheet

Three artefacts make the study reproducible and reusable: a **manifest** of everything that determines the numbers, the **results table** above, and a **datasheet** for the data (what it is, where from, what it is for, what it must not be used for). We generate them in code so that they cannot drift from the run.

In [ ]:
def sha(obj): return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:16]
manifest = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "question": "Does a small MLP beat logistic regression on 20-intent Banking77 embeddings, and does weight decay help?",
    "data": {"source": "mteb/banking77 (mirror of PolyAI/banking77, CC BY 4.0)", "intents": INTENTS, "train_per_intent": 20, "n_train": len(tr), "n_test": len(te),
             "train_ids_sha": sha(list(tr.text)), "test_ids_sha": sha(list(te.text))},
    "features": "all-MiniLM-L6-v2 sentence embeddings, L2-normalised",
    "systems": {"A": "LogisticRegression(C=10)", "B": "MLP(256), alpha=1e-6", "C": "MLP(256), alpha=0.1"},
    "seeds": SEEDS, "tuning": "none per variant (stated limitation)", "other_limitations": "MLPs reached the 300-iteration cap without converging; logistic regression is seed-independent; one fixed test sample",
    "libraries": {"python": platform.python_version(), "numpy": np.__version__, "scipy": scipy.__version__, "sklearn": sklearn.__version__},
    "results_mean_accuracy": {s: round(float(acc[s].mean()), 4) for s in "ABC"},
}
json.dump(manifest, open("study_manifest.json", "w"), indent=2)
print("manifest written:", sha(manifest)[:8], "...", len(json.dumps(manifest)), "bytes")

DATASHEET = f'''# Datasheet: 20-intent Banking77 subset used in this study
**What it is.** Customer messages to a bank's support channel, each labelled with one of 77 intents; this study uses {len(INTENTS)} intents.
**Source and licence.** PolyAI Banking77 (Casanueva et al., 2020), CC BY 4.0, read from a public mirror.
**Composition here.** {len(tr)} training messages ({tr.groupby("label_text").size().iloc[0]} per intent, fixed seed) and {len(te)} test messages (all official test messages of these intents).
**Intended use.** Comparing small classifiers on intent detection; teaching experimental method.
**Not for.** Judging a production bank assistant (the messages are short, English-only, from one collection process); inferring anything about real customers.
**Known limitations.** One language; 20 intents out of 77; labels were assigned by the dataset authors and may contain errors; test messages are a fixed sample, so results carry test-sampling uncertainty.
**Reproduce.** study_manifest.json plus this notebook.
'''
print(DATASHEET)

## Part 5 - What a paper's results section needs (a checklist)

1. **A question and a claim sized to the evidence.** "On this data, B exceeded A by x points (interval ...)", not "neural networks are better".
2. **Seeds and variance.** Number of seeds, what varied between them (initialisation, data order, data split), mean and spread.
3. **A baseline that is hard to beat** and the same tuning budget for every system.
4. **An ablation** that isolates each claimed contribution.
5. **Uncertainty**: intervals, an effect size, and a multiple-comparison correction when you ran several tests.
6. **Negative and null results reported.** What did not work, and what the study was too small to decide.
7. **Failure analysis**: read some wrong predictions and say what kind they are.
8. **Reproducibility package**: code, manifest (model and data versions or hashes, seeds, library versions, hardware), results table.
9. **Data and licence statement**, a datasheet for any dataset you release, and a benchmark card (what it measures, what it does not) for any benchmark you release.
10. **Limitations and ethics**, honestly: who could be harmed by relying on this, and what you did not test.

A **benchmark release** adds: a clear task definition and metric; a held-out test set whose labels are not public (or a plan for when it leaks, since public test sets can enter training data); baseline results with intervals; versioning, so that a number quoted a year later can be tied to a data version; and a licence that allows the use you intend.

Venues, formats and review processes differ and change; read the call for papers or the benchmark track's guidelines for the one you target, and ask someone who has published there to read a draft. This lab teaches the *evidence* a reviewer will look for. It does not substitute for the writing, the novelty, or the peer review.

## What to hand in

The results table with the three intervals for a study of your own (change the intents or add a fourth system), the completed manifest, a datasheet for the data you used, and a half-page results section that makes exactly the claims your intervals support.